In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load the dataset
df = pd.read_csv("Dataset/default of credit card clients.csv")

In [3]:
# Basic dataset overview
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nColumn Names:\n{list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nBasic Statistics:\n{df.describe()}")

Shape: (30000, 25)

First 5 rows:
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_1  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  dpnm  
0         0 

In [4]:
# Target variable distribution
print(df["dpnm"].value_counts())
print(f"Default Rate: {df['dpnm'].mean() * 100:.2f}%")

dpnm
0    23364
1     6636
Name: count, dtype: int64
Default Rate: 22.12%


In [5]:
# --- Handle Missing Values ---
print(df.isnull().sum())
df.fillna(0, inplace=True)
print("Missing values after fill:", df.isnull().sum().sum())

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dpnm         0
dtype: int64
Missing values after fill: 0


In [6]:
# ---  Drop Irrelevant Columns ---
if "ID" in df.columns:
    df.drop("ID", axis=1, inplace=True)
    print("\nDropped 'ID' column")


Dropped 'ID' column


In [7]:
# ---  Validate Categorical Values ---
invalid_sex = df[~df["SEX"].isin([1, 2])].shape[0]
print(f"Invalid SEX values: {invalid_sex}")
df = df[df["SEX"].isin([1, 2])]

Invalid SEX values: 0


In [8]:
invalid_edu = df[~df["EDUCATION"].isin([1, 2, 3, 4])].shape[0]
print(f"Invalid EDUCATION values remapped to 'others': {invalid_edu}")
df["EDUCATION"] = df["EDUCATION"].apply(lambda x: 4 if x not in [1, 2, 3, 4] else x)

Invalid EDUCATION values remapped to 'others': 345


In [9]:
invalid_mar = df[~df["MARRIAGE"].isin([1, 2, 3])].shape[0]
print(f"Invalid MARRIAGE values remapped to 'others': {invalid_mar}")
df["MARRIAGE"] = df["MARRIAGE"].apply(lambda x: 3 if x not in [1, 2, 3] else x)

Invalid MARRIAGE values remapped to 'others': 54


In [10]:
# ---  Feature Engineering ---
df["TOTAL_BILL"] = (
    df["BILL_AMT1"] + df["BILL_AMT2"] + df["BILL_AMT3"] +
    df["BILL_AMT4"] + df["BILL_AMT5"] + df["BILL_AMT6"]
)

df["TOTAL_PAYMENT"] = (
    df["PAY_AMT1"] + df["PAY_AMT2"] + df["PAY_AMT3"] +
    df["PAY_AMT4"] + df["PAY_AMT5"] + df["PAY_AMT6"]
)

df["PAYMENT_RATIO"] = df["TOTAL_PAYMENT"] / (df["TOTAL_BILL"] + 1)

df["AVG_DELAY"] = (
    df["PAY_1"] + df["PAY_2"] + df["PAY_3"] +
    df["PAY_4"] + df["PAY_5"] + df["PAY_6"]
) / 6

# Utilization ratio: how much of the credit limit is billed
df["UTILIZATION"] = df["TOTAL_BILL"] / (df["LIMIT_BAL"] + 1)

print("New features created: TOTAL_BILL, TOTAL_PAYMENT, PAYMENT_RATIO, AVG_DELAY, UTILIZATION")
print(f"\nFinal preprocessed shape: {df.shape}")

New features created: TOTAL_BILL, TOTAL_PAYMENT, PAYMENT_RATIO, AVG_DELAY, UTILIZATION

Final preprocessed shape: (30000, 29)


In [11]:
# ----- MODEL BUILDING -----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [12]:
# ---  Split Features and Target ---
X = df.drop("dpnm", axis=1)
y = df["dpnm"]

In [13]:
print(f"\nFeatures: {X.shape[1]} columns")
print(f"Target distribution:\n{y.value_counts()}")


Features: 28 columns
Target distribution:
dpnm
0    23364
1     6636
Name: count, dtype: int64


In [14]:
# ---  Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")


Train size: 24000 | Test size: 6000


In [15]:
# --- Handle Class Imbalance with SMOTE ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"After  SMOTE: {dict(zip(*np.unique(y_train_smote, return_counts=True)))}")

Before SMOTE: {np.int64(0): np.int64(18691), np.int64(1): np.int64(5309)}
After  SMOTE: {np.int64(0): np.int64(18691), np.int64(1): np.int64(18691)}


In [16]:
# ---  Feature Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled  = scaler.transform(X_test)

In [17]:
# --- 3.5 Train Models ---
# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_smote)
print("Logistic Regression trained.")


Logistic Regression trained.


In [18]:
# Model 2: Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf.fit(X_train_scaled, y_train_smote)
print("Random Forest trained.")

Random Forest trained.


In [19]:
# Model 3: XGBoost
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric="logloss", verbosity=0
)
xgb.fit(X_train_scaled, y_train_smote)
print("XGBoost trained.")

XGBoost trained.


In [20]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import GridSearchCV, cross_val_score
import seaborn as sns
import matplotlib.pyplot as plt

In [21]:
# ---  Hyperparameter Tuning (XGBoost with GridSearch) ---
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [6, 8],
    "learning_rate": [0.05, 0.1],
}

In [22]:
grid_search = GridSearchCV(
    XGBClassifier(subsample=0.8, colsample_bytree=0.8,
                  random_state=42, eval_metric="logloss", verbosity=0),
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0
)

In [23]:
grid_search.fit(X_train_scaled, y_train_smote)
print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

Best Params: {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 300}
Best CV ROC-AUC: 0.9149


In [24]:
# Use the best XGBoost model
best_xgb = grid_search.best_estimator_

In [25]:
# ---  Cross-Validation ---
cv_scores = cross_val_score(best_xgb, X_train_scaled, y_train_smote, cv=5, scoring="roc_auc")
print(f"CV ROC-AUC Scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")

CV ROC-AUC Scores: [0.7933 0.9107 0.9853 0.9843 0.9851]
Mean: 0.9318 | Std: 0.0749


In [26]:
# ---  Full Model Evaluation ---
def evaluate_model(model, X_test, y_test, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)
    f1  = f1_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec  = recall_score(y_test, pred)
    cm   = confusion_matrix(y_test, pred)

    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, pred)}")

    # Confusion matrix plot
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Default", "Default"],
                yticklabels=["No Default", "Default"])
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{name.replace(' ', '_')}.png", dpi=100)
    plt.close()
    print(f"  Saved confusion matrix: confusion_matrix_{name.replace(' ', '_')}.png")

    return {"Model": name, "Accuracy": acc, "ROC_AUC": auc,
            "F1": f1, "Precision": prec, "Recall": rec}

In [27]:
print("\n---  Model Evaluation Results ---")
lr_metrics  = evaluate_model(lr,       X_test_scaled, y_test, "Logistic Regression")
rf_metrics  = evaluate_model(rf,       X_test_scaled, y_test, "Random Forest")
xgb_metrics = evaluate_model(best_xgb, X_test_scaled, y_test, "XGBoost (Tuned)")


---  Model Evaluation Results ---

  Logistic Regression
  Accuracy  : 0.6967
  ROC-AUC   : 0.6881
  F1-Score  : 0.4502
  Precision : 0.3757
  Recall    : 0.5614

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.74      0.79      4673
           1       0.38      0.56      0.45      1327

    accuracy                           0.70      6000
   macro avg       0.62      0.65      0.62      6000
weighted avg       0.75      0.70      0.72      6000

  Saved confusion matrix: confusion_matrix_Logistic_Regression.png

  Random Forest
  Accuracy  : 0.7842
  ROC-AUC   : 0.7631
  F1-Score  : 0.5255
  Precision : 0.5114
  Recall    : 0.5403

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      4673
           1       0.51      0.54      0.53      1327

    accuracy                           0.78      6000
   macro avg       0.69      0.70      0.69      6000
w

In [28]:
# Comparison table
metrics_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])
print("\n--- Model Comparison ---")
print(metrics_df.to_string(index=False))


--- Model Comparison ---
              Model  Accuracy  ROC_AUC       F1  Precision   Recall
Logistic Regression  0.696667 0.688061 0.450151   0.375693 0.561417
      Random Forest  0.784167 0.763126 0.525467   0.511412 0.540317
    XGBoost (Tuned)  0.794000 0.754200 0.478481   0.543624 0.427280


In [29]:
best_model_name = metrics_df.sort_values("ROC_AUC", ascending=False).iloc[0]["Model"]
print(f"\nBest Model (by ROC-AUC): {best_model_name}")


Best Model (by ROC-AUC): Random Forest


In [30]:
# ROC Curve comparison
from sklearn.metrics import roc_curve
plt.figure(figsize=(8, 6))
for model, name in [(lr, "Logistic Regression"), (rf, "Random Forest"), (best_xgb, "XGBoost (Tuned)")]:
    prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curve_comparison.png", dpi=100)
plt.close()
print("\nSaved ROC curve: roc_curve_comparison.png")


Saved ROC curve: roc_curve_comparison.png


In [31]:
# Feature importance (XGBoost)
feat_imp = pd.Series(best_xgb.feature_importances_, index=X.columns)
top10 = feat_imp.nlargest(10)
plt.figure(figsize=(8, 5))
top10.sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 10 Feature Importances — XGBoost")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=100)
plt.close()
print("Saved feature importance chart: feature_importance.png")

Saved feature importance chart: feature_importance.png


In [32]:
import joblib
# ---  Save Models ---
joblib.dump(best_xgb, "credit_default_model.pkl")
joblib.dump(scaler,   "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")
print("\nModels saved: credit_default_model.pkl, scaler.pkl, feature_columns.pkl")



Models saved: credit_default_model.pkl, scaler.pkl, feature_columns.pkl


In [33]:
# ---  Load and Use Saved Model ---
loaded_model  = joblib.load("credit_default_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

In [34]:
# Use a sample from X_train (real training customer, not synthetic SMOTE)
sample = X_train.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)

In [35]:
probability = loaded_model.predict_proba(sample_scaled)[0][1]
prediction  = loaded_model.predict(sample_scaled)[0]

if probability >= 0.70:
    risk_level = "HIGH"
elif probability >= 0.40:
    risk_level = "MEDIUM"
else:
    risk_level = "LOW"

In [36]:
print(f"  Prediction  : {'DEFAULT' if prediction == 1 else 'NO DEFAULT'}")
print(f"  Probability : {probability:.4f}")
print(f"  Risk Level  : {risk_level}")


  Prediction  : NO DEFAULT
  Probability : 0.1701
  Risk Level  : LOW


In [37]:
# --- Batch Risk Report for All Test Customers ---

X_test_scaled_full = loaded_scaler.transform(X_test)
predictions  = loaded_model.predict(X_test_scaled_full)
probabilities = loaded_model.predict_proba(X_test_scaled_full)[:, 1]

In [38]:
risk_levels = np.where(
    probabilities >= 0.70, "HIGH",
    np.where(probabilities >= 0.40, "MEDIUM", "LOW")
)

results_df = pd.DataFrame({
    "Actual_Default"    : y_test.values,
    "Predicted_Default" : predictions,
    "Risk_Probability"  : probabilities.round(4),
    "Risk_Level"        : risk_levels
})

results_df.to_csv("customer_risk_report.csv", index=False)
print(f"  Saved {len(results_df)} customer predictions to: customer_risk_report.csv")



  Saved 6000 customer predictions to: customer_risk_report.csv


In [39]:
# Risk level distribution
print("\n  Risk Level Distribution:")
print(pd.Series(risk_levels).value_counts().to_string())


  Risk Level Distribution:
LOW       4629
MEDIUM     798
HIGH       573


In [40]:
#---  Application Summary Dashboard ---
high   = (risk_levels == "HIGH").sum()
medium = (risk_levels == "MEDIUM").sum()
low    = (risk_levels == "LOW").sum()
total  = len(risk_levels)



In [41]:
print(f"  Total Customers Scored : {total}")
print(f"  HIGH Risk  (>=70%)     : {high}  ({high/total*100:.1f}%)")
print(f"  MEDIUM Risk(40%-70%)   : {medium} ({medium/total*100:.1f}%)")
print(f"  LOW Risk   (<40%)      : {low}  ({low/total*100:.1f}%)")



  Total Customers Scored : 6000
  HIGH Risk  (>=70%)     : 573  (9.6%)
  MEDIUM Risk(40%-70%)   : 798 (13.3%)
  LOW Risk   (<40%)      : 4629  (77.1%)


In [42]:
plt.figure(figsize=(6, 4))
pd.Series(risk_levels).value_counts().plot(
    kind="bar",
    color=["#d9534f", "#f0ad4e", "#5cb85c"],
    edgecolor="black"
)
plt.title("Customer Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("risk_distribution.png", dpi=100)
plt.close()
print("\n  Saved risk distribution chart: risk_distribution.png")


  Saved risk distribution chart: risk_distribution.png
